# 📝 Домашнє завдання — Урок 2 (Архітектура, SOLID, Repository)

**До уроку:** [lektsiya-2-arkhitektura-solid-repository.ipynb](lektsiya-2-arkhitektura-solid-repository.ipynb)

Пишіть розв'язок у порожню комірку під кожним завданням; запускайте `Shift + Enter`.

### Завдання 1 — @property (обчислюваний)

Створіть клас `Rectangle(width, height)` з властивістю `area` (через `@property`), що рахує площу. Змініть `width` і переконайтесь, що площа перерахувалась. _(Розділ 1.)_

In [2]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        
    @property
    def area(self):
        return self.width * self.height
    
rec1 = Rectangle(7,5)
print("Area:",rec1.area)
rec2 = Rectangle(4,5)
print("Area:",rec2.area)

Area: 35
Area: 20


### Завдання 2 — @property з валідацією

Створіть клас `Product` із властивістю `price`: сеттер не дозволяє від'ємну ціну (кидає `ValueError`). Продемонструйте коректний і помилковий випадки. _(Розділ 1.)_

In [6]:
class Product:
    def __init__(self, name, price):
        self.name = name
        self._price = price
        
    @property
    def price(self):
        return self._price
    
    @price.setter
    def price(self, value):
        if value < 0:
            raise ValueError("Price must be a positive number")
        self._price = value
        
prod1 = Product("Apple", 15)
print("Price:", prod1.price)
prod1.price = 25
print("Updated Price:", prod1.price)
try:
    prod1.price = -5
except ValueError as e:
    print(e)

Price: 15
Updated Price: 25
Price must be a positive number


### Завдання 3 — Композиція

Створіть клас `Computer`, що МАЄ об'єкт `CPU` (композиція). Метод `Computer.run()` делегує роботу CPU. _(Розділ 2.)_

In [11]:
class CPU:
    def method(self):
        return f"CPU method called"

class Computer:
    def __init__(self, ram):
        self.ram = ram
        self.cpu = CPU()
        
    def run(self):
        return f"Ram: {self.ram}, cpu: {self.cpu.method()}"
    
    
comp = Computer(16)
print(comp.run())

Ram: 16, cpu: CPU method called


### Завдання 4 — Single Responsibility

Дано клас, що і рахує зарплату, і друкує звіт. Розділіть його на два класи (`SalaryCalculator`, `ReportPrinter`). _(Розділ 3.)_

In [12]:
class SalaryCalculator:
    def calculate(self, salary, hours):
        return salary * hours
    
class ReportPrinter:
    def report(self,salary):
        return f"Report: {salary}"
    
sc = SalaryCalculator()
a = sc.calculate(160, 80)
rp = ReportPrinter()
print(rp.report(a))

Report: 12800


### Завдання 5 — Абстракція / Dependency Inversion

Створіть абстрактний `PaymentMethod` з методом `pay(amount)` та дві реалізації (`CardPayment`, `CashPayment`). Напишіть функцію `checkout(method, amount)`, що працює з будь-якою реалізацією. _(Розділ 3.)_

In [14]:
from abc import ABC, abstractmethod

class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        """..."""
        
class CardPayment(PaymentMethod):
    def pay(self, amount):
        if amount < 0:
            raise ValueError("Amount must be positive")
        return f"Card payment successful"
    
class CashPayment(PaymentMethod):
    def pay(self, amount):
        return f"Cash payment successful"
    
def payment(method: PaymentMethod, amount):
    return method.pay(amount)


print("Card payment:", payment(CardPayment(), 200))
print("Cash payment:", payment(CashPayment(), 400))

Card payment: Card payment successful
Cash payment: Cash payment successful


### Завдання 6 — Repository

Реалізуйте `InMemoryUserRepository` з методами `add(user)`, `get(user_id)`, `list()`. Користувач — простий об'єкт з `id` та `name`. _(Розділ 5.)_

In [ ]:
from abc import ABC, abstractmethod

class User:
    def __init__(self, name, id=None):
        self.name = name
        self.id = id
    def __repr__(self):
        return f"User {self.id}, name {self.name}"
    

class AbstractUserRepository(ABC):
    @abstractmethod
    def add(self, user):
        "..."
    @abstractmethod
    def get(self,user_id):
        "..."
    @abstractmethod
    def list(self):
        "..."
        
class InMemoryUserRepository(AbstractUserRepository):
    def __init__(self):
        self._next_id = 1
        self.items = {}
    def add(self, user):
        if user.id is None:
            user.id = self._next_id
        self.items[user.id] = user
        self._next_id = max(self._next_id, user.id + 1)
        return user
    def get(self, user_id):
        return self.items.get(user_id)
    def list(self):
        return list(self.items.values())

### Завдання 7 — Сервісний шар

Напишіть `UserService`, що приймає репозиторій у конструкторі та має метод `register(name)` (створює й зберігає користувача). Продемонструйте з репозиторієм із завдання 6. _(Розділ 6.)_

In [16]:
from abc import ABC, abstractmethod

class User:
    def __init__(self, name, id=None):
        self.name = name
        self.id = id
    def __repr__(self):
        return f"User: {self.id}, name: {self.name}"
    

class AbstractUserRepository(ABC):
    @abstractmethod
    def add(self, user):
        "..."
    @abstractmethod
    def get(self,user_id):
        "..."
    @abstractmethod
    def list(self):
        "..."


class InMemoryUserRepository(AbstractUserRepository):
    def __init__(self):
        self._next_id = 1
        self.items = {}
    def add(self, user):
        if user.id is None:
            user.id = self._next_id
        self.items[user.id] = user
        self._next_id = max(self._next_id, user.id + 1)
        return user
    def get(self, user_id):
        return self.items.get(user_id)
    def list(self):
        return list(self.items.values())


class UserService:
    def __init__(self, repository):
        self._repo = repository
    def register(self, name):
        return self._repo.add(User(name))
    
    
repo = InMemoryUserRepository()
service = UserService(repo)
user1 = service.register("Anna")
user2 = service.register("Asia")

print(repo.list())

[User: 1, name: Anna, User: 2, name: Asia]


### Завдання 8 — Змінні середовища

За допомогою `os.getenv` прочитайте змінну `APP_NAME` (за замовчуванням `"myapp"`) та `DEBUG` (перетворіть на `bool`). Виведіть їх. _(Розділ 9.)_

In [17]:
import os

app_name = os.getenv("APP_NAME", "myapp")
debug = os.getenv("DEBUG", "false").lower() == "true"

print(f"App name: {app_name}")
print(f"Debug: {debug}")

App name: myapp
Debug: False


### Завдання 9 (⭐) — Розширити скелет

Відкрийте [skelet-proyektu](skelet-proyektu/) і додайте у `TaskService` метод `delete_task(task_id)` (+ метод `delete` у репозиторії). Запустіть тести. _(Розділи 5–7.)_

In [ ]:
#Тут додаю тільки delete_task з TaskService. Решта у репозиторії і в тестах(пройшли успішно)
def delete_task(self, task_id: int) -> None:
        task = self._repo.get(task_id)
        if task is None:
            raise TaskNotFoundError(f"Завдання {task_id} не знайдено")
        self._repo.delete(task_id)

---
🎉 Це остання домашка курсу. Вітаємо із завершенням — успіхів на співбесідах!